# MedGemma QLoRA with Unsloth on Kaggle T4 - Low-Memory v26

This notebook fine-tunes an Unsloth 4-bit MedGemma 1.5 4B model for 9-class colorectal histology patch classification using NCT-CRC-HE-100K and evaluates on CRC-VAL-HE-7K.

v26 keeps the working repo/token/HF-transfer/eager-attention/no-TorchAO fixes and adds a safer SigLIP LayerNorm patch. The previous run reached the first training step but failed in the Gemma 3 / MedGemma SigLIP vision tower with `RuntimeError: expected scalar type Half but found Float`. The new patch computes vision LayerNorm in float32 and casts its output back to the incoming activation dtype, and it monkey-patches `model.for_training(...)` so vision gradient-checkpointing does not get re-enabled by the trainer wrapper.

Run the dependency cell first. It may restart the kernel once. After restart, run from the top again.

Recommended Kaggle accelerator: **T4 x2**. The notebook uses a single process, so it will not automatically train on both T4s, but T4 is friendlier to this Unsloth/Triton/Gemma 3 stack than P100.


**v26 fix:** disables Unsloth/TorchDynamo auto-compilation to avoid `ArgsMismatchError: missing a required argument: x` in the Gemma3 `q_norm` compile path on Kaggle.

## 1. Dependency bootstrap for Kaggle P100 or T4


In [ ]:
# Run this cell first, before importing torch / transformers / datasets / unsloth.
# It repairs Kaggle's Python environment, validates the exact package stack,
# then restarts the kernel once so imports reload cleanly.

import os
import sys
import subprocess
from pathlib import Path
from importlib import metadata

# Kaggle can inherit HF_HUB_ENABLE_HF_TRANSFER=1 without the optional hf_transfer package.
# Disable it before any huggingface_hub import; otherwise hf_hub_download can fail before model loading.
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

# Stability flags for Kaggle + Gemma3/MedGemma vision training.
# Must be set before importing unsloth/transformers/torch where possible.
os.environ['UNSLOTH_COMPILE_DISABLE'] = '1'
os.environ['UNSLOTH_COMPILE_IGNORE_ERRORS'] = '1'
os.environ['UNSLOTH_FULLGRAPH'] = '0'
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['TORCH_COMPILE_DISABLE'] = '1'

# Optional for Kaggle T4 x2 single-notebook runs. Uncomment to force one GPU.
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'

MARKER = Path('/kaggle/working/.medgemma_unsloth_deps_v26_installed')
CORE_MODULES = (
    'torch', 'numpy', 'pandas', 'scipy', 'sklearn', 'transformers',
    'datasets', 'huggingface_hub', 'tokenizers', 'peft', 'trl', 'unsloth', 'unsloth_zoo', 'torchao'
)

# Use the current Unsloth Gemma 3 notebook-compatible stack. Transformers 4.56.x
# Gemma 3 mask creation uses or_mask_function/and_mask_function, which requires
# torch >= 2.6. Pin a T4-compatible CUDA 12.4 Torch 2.6 stack.
PINNED = {
    'torch': '2.6.0',
    'torchvision': '0.21.0',
    'torchaudio': '2.6.0',
    'transformers': '4.56.2',
    'tokenizers': '0.22.1',
    'trl': '0.22.2',
    'peft': '0.18.0',
    'datasets': '4.3.0',
    'huggingface-hub': '0.36.0',
    'numpy': '1.26.4',
    'pandas': '2.2.2',
    'matplotlib': '3.8.4',
    'pillow': '11.3.0',
}


def pip_install(args):
    print('+', ' '.join(args), flush=True)
    subprocess.check_call(args)


def pkg_version(*names):
    for name in names:
        try:
            return metadata.version(name)
        except metadata.PackageNotFoundError:
            pass
    return None


def version_tuple(v):
    parts = []
    for part in v.split('+')[0].split('.')[:3]:
        m = ''.join(ch for ch in part if ch.isdigit())
        if m:
            parts.append(int(m))
    return tuple(parts)


def hub_ok(v):
    if v is None:
        return False
    t = version_tuple(v)
    return (0, 30, 0) <= t < (1, 0, 0)


def active_runtime_ok():
    try:
        import torch
        arch = torch.cuda.get_arch_list() if torch.cuda.is_available() else []
        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
        capability = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None

        versions = {
            'torch': pkg_version('torch'),
            'torchvision': pkg_version('torchvision'),
            'torchaudio': pkg_version('torchaudio'),
            'transformers': pkg_version('transformers'),
            'tokenizers': pkg_version('tokenizers'),
            'trl': pkg_version('trl'),
            'peft': pkg_version('peft'),
            'datasets': pkg_version('datasets'),
            'huggingface-hub': pkg_version('huggingface-hub', 'huggingface_hub'),
            'unsloth': pkg_version('unsloth'),
            'unsloth-zoo': pkg_version('unsloth-zoo', 'unsloth_zoo'),
            'torchao': pkg_version('torchao'),
        }

        is_p100 = capability == (6, 0)
        torch_arch_ok = ('sm_60' in arch) if is_p100 else True

        print('GPU:', gpu_name)
        print('GPU capability:', capability)
        print('Torch CUDA runtime:', torch.version.cuda)
        print('Torch CUDA arch list:', arch)
        for k, v in versions.items():
            print(f'{k}:', v)

        return (
            versions['torch'] is not None and versions['torch'].startswith(PINNED['torch'])
            and torch_arch_ok
            and versions['transformers'] == PINNED['transformers']
            and versions['tokenizers'] == PINNED['tokenizers']
            and versions['trl'] == PINNED['trl']
            and versions['peft'] == PINNED['peft']
            and versions['datasets'] == PINNED['datasets']
            and hub_ok(versions['huggingface-hub'])
            and versions['unsloth'] is not None
            and versions['unsloth-zoo'] is not None
            and versions['torchao'] is None
        )
    except Exception as exc:
        print('Runtime validation failed:', repr(exc))
        return False


if MARKER.exists() and active_runtime_ok():
    print('Dependency check passed.')
else:
    if MARKER.exists():
        print('Marker exists, but runtime is not compatible. Reinstalling.')
    else:
        print('Installing MedGemma/Unsloth QLoRA stack for v26.')

    already_loaded = [m for m in CORE_MODULES if m in sys.modules]
    if already_loaded:
        print('Already loaded in this kernel:', ', '.join(already_loaded))
        print('Installing now, then restarting so Python reloads packages cleanly.')

    # Remove any stale TorchAO first; it can break importing transformers with Torch 2.6.0.
    try:
        pip_install([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
    except Exception as exc:
        print('Initial torchao uninstall failed or torchao was not installed; continuing:', repr(exc))

    # Install a CUDA 12.4 Torch 2.6 stack. Torch 2.5 reaches Gemma 3 training but
    # fails in transformers.masking_utils because Gemma 3 uses mask combinators
    # that require torch>=2.6. Torch 2.11 is still avoided because Unsloth pins <2.11.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
        'torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0',
        '--index-url', 'https://download.pytorch.org/whl/cu124'
    ])

    # Scientific stack. --no-deps prevents pip from upgrading NumPy/Pandas behind our back.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
        'numpy==1.26.4', 'pandas==2.2.2', 'matplotlib==3.8.4', 'pillow==11.3.0', 'tqdm'
    ])

    # Hugging Face + training stack. These versions align with Unsloth's current Gemma 3 notebook:
    # transformers 4.56.2 requires tokenizers >=0.22,<0.23; pin tokenizers explicitly.
    # Keep hub < 1.0 for transformers 4.x.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
        'transformers==4.56.2',
        'tokenizers==0.22.1',
        'trl==0.22.2',
        'peft==0.18.0',
        'datasets==4.3.0',
        'huggingface-hub==0.36.0',
        'accelerate==1.9.0',
        'bitsandbytes==0.46.1',
        'sentencepiece', 'protobuf', 'safetensors', 'packaging', 'hf_xet>=1.1.0,<2.0',
        'hf_transfer', 'tyro', 'msgspec', 'cut_cross_entropy'
    ])

    # Unsloth packages without dependency resolution so they cannot pull a newer Torch.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
        'unsloth', 'unsloth_zoo'
    ])

    # Match xFormers to torch 2.6.0. If this fails, continue; eager attention is used for SigLIP.
    try:
        pip_install([
            sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
            'xformers==0.0.29.post3'
        ])
    except Exception as exc:
        print('xformers install failed; continuing without xformers:', repr(exc))

    # TorchAO is intentionally removed. This notebook uses bitsandbytes 4-bit loading,
    # not TorchAO quantization, so leaving torchao installed only adds another failure mode.
    try:
        pip_install([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
    except Exception as exc:
        print('torchao uninstall failed or torchao was not installed; continuing:', repr(exc))

    MARKER.write_text('installed')
    print('\nDependency repair complete. Restarting kernel now.')
    print('After Kaggle restarts, run the notebook from the top again.')
    os._exit(0)


## 2. Imports, constants, metrics helpers


In [ ]:
import os
import re
import json
import time
import math
import shutil
import zipfile
import random
import urllib.request
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Important: set before importing huggingface_hub / transformers / unsloth.
# Kaggle may set this to 1 globally, but hf_transfer is optional and often absent.
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
# Disable Unsloth/TorchDynamo compilation for this Kaggle Gemma3 vision run.
# This avoids ArgsMismatchError in unsloth_zoo temporary Gemma patches.
os.environ['UNSLOTH_COMPILE_DISABLE'] = '1'
os.environ['UNSLOTH_COMPILE_IGNORE_ERRORS'] = '1'
os.environ['UNSLOTH_FULLGRAPH'] = '0'
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['TORCH_COMPILE_DISABLE'] = '1'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:64')

import numpy as np
import pandas as pd
import torch

# Extra guard: even with env vars, make TorchDynamo non-fatal if an imported library touches it.
try:
    import torch._dynamo as _torch_dynamo
    _torch_dynamo.config.suppress_errors = True
    try:
        _torch_dynamo.disable()
    except Exception:
        pass
    print('TorchDynamo disabled/suppressed for stable Kaggle training.')
except Exception as exc:
    print('TorchDynamo guard unavailable:', repr(exc))
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

# TorchAO is intentionally absent for this Kaggle Torch 2.5.1 stack.
# If it is still installed, transformers may crash while importing torchao before Unsloth starts.
try:
    from importlib import metadata as _guard_metadata
    _torchao_version = _guard_metadata.version('torchao')
    raise RuntimeError(
        f'torchao=={_torchao_version} is installed, but this notebook intentionally removes torchao. '
        'Run the dependency bootstrap cell first, allow the restart, then run from the top.'
    )
except _guard_metadata.PackageNotFoundError:
    print('TorchAO: not installed (intentional for this Kaggle Torch 2.5.1 stack)')

# Import Unsloth before datasets/transformers so its patches are applied.
from unsloth import FastVisionModel, FastModel, get_chat_template

from huggingface_hub import login, hf_hub_download, whoami
from datasets import load_dataset
from transformers import AutoConfig

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU compute capability:', torch.cuda.get_device_capability(0))
    print('Torch:', torch.__version__)
    print('Torch arch list:', torch.cuda.get_arch_list())
    if torch.cuda.get_device_capability(0) == (6, 0) and 'sm_60' not in torch.cuda.get_arch_list():
        raise RuntimeError('Active torch does not include sm_60 kernels. Rerun the dependency cell and restart.')


try:
    from importlib import metadata as importlib_metadata
    print('Transformers:', importlib_metadata.version('transformers'))
    print('Tokenizers:', importlib_metadata.version('tokenizers'))
    print('Hugging Face Hub:', importlib_metadata.version('huggingface-hub'))
    print('Datasets:', importlib_metadata.version('datasets'))
    print('Unsloth:', importlib_metadata.version('unsloth'))
    print('Unsloth Zoo:', importlib_metadata.version('unsloth-zoo'))
    try:
        print('TorchAO:', importlib_metadata.version('torchao'))
    except importlib_metadata.PackageNotFoundError:
        print('TorchAO: not installed')
except Exception as exc:
    print('Package version diagnostics unavailable:', repr(exc))

PROJECT_DIR = Path('/kaggle/working/MedGemma_Unsloth_QLoRA_CRC_lowmem_v26')
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
ADAPTER_DIR = PROJECT_DIR / 'medgemma_crc_unsloth_lora'
for p in [PROJECT_DIR, DATA_DIR, RESULTS_DIR, ADAPTER_DIR]:
    p.mkdir(parents=True, exist_ok=True)

LABEL_NAMES = ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
LABEL_DESCRIPTIONS = {
    'ADI': 'adipose tissue / fat',
    'BACK': 'empty background / no tissue',
    'DEB': 'debris / necrotic debris',
    'LYM': 'lymphocytes / lymphoid cells',
    'MUC': 'mucus / mucin pools',
    'MUS': 'smooth muscle',
    'NORM': 'normal colon mucosa',
    'STR': 'cancer-associated stroma',
    'TUM': 'colorectal adenocarcinoma epithelium / tumor epithelium',
}
LABEL_TO_DIGIT = {label: str(i) for i, label in enumerate(LABEL_NAMES)}
DIGIT_TO_LABEL = {str(i): label for i, label in enumerate(LABEL_NAMES)}

CLASS_INSTRUCTION = """Classify this H&E colorectal histology image patch.
Return exactly one digit and nothing else.

0 = ADI, adipose tissue / fat
1 = BACK, empty background / no tissue
2 = DEB, debris / necrotic debris
3 = LYM, lymphocytes / lymphoid cells
4 = MUC, mucus / mucin pools
5 = MUS, smooth muscle
6 = NORM, normal colon mucosa
7 = STR, cancer-associated stroma
8 = TUM, colorectal adenocarcinoma epithelium / tumor epithelium

Answer with only one digit from 0 to 8."""


def simple_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred)) if len(y_true) else 0.0


def simple_confusion_matrix(y_true, y_pred, labels):
    label_to_pos = {label: idx for idx, label in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)
    for t, p in zip(y_true, y_pred):
        if t in label_to_pos and p in label_to_pos:
            cm[label_to_pos[t], label_to_pos[p]] += 1
    return cm


def classification_report_df(y_true, y_pred, labels):
    cm = simple_confusion_matrix(y_true, y_pred, labels)
    rows = []
    total = cm.sum()
    correct = np.trace(cm)
    for i, label in enumerate(labels):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        support = cm[i, :].sum()
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({'label': label, 'precision': precision, 'recall': recall, 'f1-score': f1, 'support': int(support)})
    macro_f1 = float(np.mean([r['f1-score'] for r in rows])) if rows else 0.0
    weighted_f1 = float(sum(r['f1-score'] * r['support'] for r in rows) / total) if total else 0.0
    acc = float(correct / total) if total else 0.0
    rows.append({'label': 'accuracy', 'precision': acc, 'recall': acc, 'f1-score': acc, 'support': int(total)})
    rows.append({'label': 'macro avg', 'precision': np.mean([r['precision'] for r in rows[:len(labels)]]), 'recall': np.mean([r['recall'] for r in rows[:len(labels)]]), 'f1-score': macro_f1, 'support': int(total)})
    rows.append({'label': 'weighted avg', 'precision': 0.0, 'recall': 0.0, 'f1-score': weighted_f1, 'support': int(total)})
    return pd.DataFrame(rows)


def plot_confusion_matrix(cm, labels, title, output_path):
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation='nearest')
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=labels,
        yticklabels=labels,
        ylabel='True label',
        xlabel='Predicted label',
        title=title,
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
    threshold = cm.max() / 2 if cm.size and cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center')
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## 3. Hugging Face login from Kaggle Secrets


In [ ]:
def get_hf_token():
    token = os.environ.get('HF_TOKEN', '').strip()
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
        if token:
            return token.strip()
    except Exception:
        pass
    print(
        'HF_TOKEN not found. Continuing anonymously for the public Unsloth mirror. '        'If you switch back to google/medgemma-1.5-4b-it, add a Kaggle secret named HF_TOKEN '        'from the same Hugging Face account that accepted the gated model terms.'
    )
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    try:
        print('Hugging Face account:', whoami(token=HF_TOKEN).get('name'))
    except Exception as exc:
        print('HF login worked, but whoami failed:', repr(exc))
else:
    print('No Hugging Face token in this runtime.')


## 4. Locate or download NCT-CRC-HE-100K and CRC-VAL-HE-7K


In [ ]:
NCT_CRC_HE_100K_URL = 'https://zenodo.org/records/1214456/files/NCT-CRC-HE-100K.zip'
CRC_VAL_HE_7K_URL = 'https://zenodo.org/records/1214456/files/CRC-VAL-HE-7K.zip'


def looks_like_crc_folder(path: Path) -> bool:
    return path.is_dir() and all((path / label).is_dir() for label in LABEL_NAMES)


def find_crc_folder(folder_name: str):
    roots = [Path('/kaggle/input'), DATA_DIR, Path('/kaggle/working')]
    for root in roots:
        if not root.exists():
            continue
        for candidate in root.glob(f'**/{folder_name}'):
            if looks_like_crc_folder(candidate):
                return candidate
    # Sometimes Kaggle inputs have class folders directly under a dataset folder.
    if folder_name == 'NCT-CRC-HE-100K':
        for candidate in Path('/kaggle/input').glob('*') if Path('/kaggle/input').exists() else []:
            if looks_like_crc_folder(candidate) and 'VAL' not in candidate.name.upper():
                return candidate
    return None


def download_and_extract(url, zip_name, expected_folder):
    existing = find_crc_folder(expected_folder)
    if existing is not None:
        print(f'Found {expected_folder}:', existing)
        return existing

    zip_path = DATA_DIR / zip_name
    if not zip_path.exists():
        print('Downloading:', url)
        print('This may take several minutes for NCT-CRC-HE-100K.')
        urllib.request.urlretrieve(url, zip_path)
        print('Downloaded:', zip_path)
    else:
        print('Using existing zip:', zip_path)

    print('Extracting:', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)

    extracted = find_crc_folder(expected_folder)
    if extracted is None:
        raise FileNotFoundError(f'Could not locate extracted {expected_folder}. Check zip layout under {DATA_DIR}.')
    print(f'Found {expected_folder}:', extracted)
    return extracted

TRAIN_DIR = download_and_extract(NCT_CRC_HE_100K_URL, 'NCT-CRC-HE-100K.zip', 'NCT-CRC-HE-100K')
TEST_DIR = download_and_extract(CRC_VAL_HE_7K_URL, 'CRC-VAL-HE-7K.zip', 'CRC-VAL-HE-7K')

train_all = load_dataset('imagefolder', data_dir=str(TRAIN_DIR), split='train')
test_dataset = load_dataset('imagefolder', data_dir=str(TEST_DIR), split='train')

train_label_names = train_all.features['label'].names
test_label_names = test_dataset.features['label'].names
print('Train samples:', len(train_all), train_label_names)
print('Test samples:', len(test_dataset), test_label_names)

if train_label_names != LABEL_NAMES:
    print('Warning: dataset label order differs from expected LABEL_NAMES.')
if test_label_names != LABEL_NAMES:
    print('Warning: test label order differs from expected LABEL_NAMES.')


## 5. Training configuration


In [ ]:
# Low-memory Kaggle P100/T4 settings.
# This run is intended to avoid OOM first. Once it works, increase TRAIN_PER_CLASS and MAX_STEPS.

# Default to Unsloth's 4-bit MedGemma repo instead of the gated Google repo.
# The Google repo can list files publicly, but actual downloads can fail in Kaggle if token access is not approved.
MEDGEMMA_MODEL_ID = 'unsloth/medgemma-1.5-4b-it-unsloth-bnb-4bit'
FALLBACK_MEDGEMMA_MODEL_ID = 'unsloth/medgemma-1.5-4b-it-bnb-4bit'
GOOGLE_GATED_MEDGEMMA_MODEL_ID = 'google/medgemma-1.5-4b-it'

# If you intentionally want to retry the gated Google repo, set:
# MEDGEMMA_MODEL_ID = GOOGLE_GATED_MEDGEMMA_MODEL_ID

# Start conservative on P100/T4 16GB.
TRAIN_PER_CLASS = 800           # 800 x 9 = 7,200 balanced training examples. Increase to 1500+ only after this works.
MAX_STEPS = 300                 # Increase to 1200-2000 after the smoke test and validation improve.
NUM_TRAIN_EPOCHS = 1            # Used only if MAX_STEPS is None.
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8 # Effective batch 8 without increasing VRAM.
LEARNING_RATE = 1e-4            # Safer than 2e-4 on small/QLoRA vision runs.
LORA_R = 8                      # Lower VRAM than r=16/32.
LORA_ALPHA = 16
SAVE_STEPS = 50                 # Frequent trainer checkpoints so an OOM/restart can resume.
LOGGING_STEPS = 10
MAX_SEQUENCE_LENGTH = 768       # Lower than 2048 to reduce activation memory.

# Gemma 3 / MedGemma vision dtype controls.
# Default fix for the Half/Float LayerNorm crash: do NOT upcast only LayerNorms.
# Instead, feed SigLIP pixel_values in the same dtype as the vision tower and disable
# gradient checkpointing inside the frozen/non-primary vision tower.
FINETUNE_VISION_LAYERS = False          # safer on Kaggle T4; language/projector LoRA still learns from image features.
CAST_PIXEL_VALUES_TO_VISION_DTYPE = True
DISABLE_VISION_GRADIENT_CHECKPOINTING = True
FORCE_FULL_VISION_FLOAT32 = False       # emergency fallback; slower/more VRAM. Set True only if dtype crash persists.
PATCH_SIGLIP_LAYERNORM_SAFE_FP32 = True # compute vision LayerNorm in fp32, return original activation dtype.
PATCH_FOR_TRAINING_REAPPLY_VISION_FIXES = True # trainer calls model.for_training(); re-apply vision fixes there.
DISABLE_UNSLOTH_COMPILE = True           # v26: avoid TorchDynamo ArgsMismatchError in Unsloth Gemma q_norm compile path.
USE_GRADIENT_CHECKPOINTING = True        # Set False only if compiler-disable still fails; may use more VRAM.

# Evaluation. Start with 1000 for a fast sanity check. Set None for the full 7,180-image benchmark.
EVAL_MAX_SAMPLES = 1000
EVAL_CHECKPOINT_EVERY = 50

print({
    'MEDGEMMA_MODEL_ID': MEDGEMMA_MODEL_ID,
    'TRAIN_PER_CLASS': TRAIN_PER_CLASS,
    'MAX_STEPS': MAX_STEPS,
    'NUM_TRAIN_EPOCHS': NUM_TRAIN_EPOCHS,
    'EVAL_MAX_SAMPLES': EVAL_MAX_SAMPLES,
    'LORA_R': LORA_R,
    'GRADIENT_ACCUMULATION_STEPS': GRADIENT_ACCUMULATION_STEPS,
    'MAX_SEQUENCE_LENGTH': MAX_SEQUENCE_LENGTH,
    'FINETUNE_VISION_LAYERS': FINETUNE_VISION_LAYERS,
    'CAST_PIXEL_VALUES_TO_VISION_DTYPE': CAST_PIXEL_VALUES_TO_VISION_DTYPE,
    'DISABLE_VISION_GRADIENT_CHECKPOINTING': DISABLE_VISION_GRADIENT_CHECKPOINTING,
    'FORCE_FULL_VISION_FLOAT32': FORCE_FULL_VISION_FLOAT32,
    'PATCH_SIGLIP_LAYERNORM_SAFE_FP32': PATCH_SIGLIP_LAYERNORM_SAFE_FP32,
    'PATCH_FOR_TRAINING_REAPPLY_VISION_FIXES': PATCH_FOR_TRAINING_REAPPLY_VISION_FIXES,
    'DISABLE_UNSLOTH_COMPILE': DISABLE_UNSLOTH_COMPILE,
    'USE_GRADIENT_CHECKPOINTING': USE_GRADIENT_CHECKPOINTING,
})


## 6. Build a balanced training subset


In [ ]:
def label_name_for_sample(sample, feature_label_names):
    return feature_label_names[int(sample['label'])]


def balanced_indices(dataset, feature_label_names, per_class, seed=SEED):
    rng = random.Random(seed)
    buckets = {label: [] for label in LABEL_NAMES}
    for idx, sample in enumerate(dataset):
        label = label_name_for_sample(sample, feature_label_names)
        if label in buckets:
            buckets[label].append(idx)
    chosen = []
    for label in LABEL_NAMES:
        idxs = buckets[label]
        rng.shuffle(idxs)
        take = len(idxs) if per_class is None else min(per_class, len(idxs))
        chosen.extend(idxs[:take])
        print(f'{label}: selected {take} / {len(idxs)}')
    rng.shuffle(chosen)
    return chosen

print('Number of labels:', len(LABEL_NAMES))
print('Digit mapping:', DIGIT_TO_LABEL)
print('Note: target digit 7 means class STR, not 7 total labels. There are 9 classes: 0 through 8.')

train_indices = balanced_indices(train_all, train_label_names, TRAIN_PER_CLASS)
train_subset = train_all.select(train_indices)
print('Training subset size:', len(train_subset))

# Save selected indices for reproducibility.
pd.DataFrame({'train_index': train_indices}).to_csv(RESULTS_DIR / 'selected_train_indices.csv', index=False)


## 7. Conversation dataset with single-token digit targets


In [ ]:
def make_user_message(include_image_placeholder=True):
    content = [{'type': 'text', 'text': CLASS_INSTRUCTION}]
    if include_image_placeholder:
        content.append({'type': 'image'})
    return {'role': 'user', 'content': content}


def convert_sample_to_conversation(sample, feature_label_names):
    label = label_name_for_sample(sample, feature_label_names)
    digit = LABEL_TO_DIGIT[label]
    image = sample['image']
    if hasattr(image, 'convert'):
        image = image.convert('RGB')
    return {
        'messages': [
            {
                'role': 'user',
                'content': [
                    {'type': 'text', 'text': CLASS_INSTRUCTION},
                    {'type': 'image', 'image': image},
                ],
            },
            {'role': 'assistant', 'content': [{'type': 'text', 'text': digit}]},
        ]
    }


class CRCConversationDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, feature_label_names):
        self.hf_dataset = hf_dataset
        self.feature_label_names = feature_label_names

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        return convert_sample_to_conversation(self.hf_dataset[int(idx)], self.feature_label_names)


train_conversations = CRCConversationDataset(train_subset, train_label_names)
print('Example training record:')
example = train_conversations[0]
print(example['messages'][0]['content'][0]['text'][:300] + '...')
target_digit = example['messages'][1]['content'][0]['text']
target_label = DIGIT_TO_LABEL[target_digit]
print(f'target digit: {target_digit} -> {target_label}')
print('Reminder: labels are zero-indexed, so digit 7 is the 8th class, STR.')


## 8. Load MedGemma with Unsloth and add QLoRA adapters


In [ ]:
# This cell is intentionally strict: if the model repo cannot provide config.json,
# stop here with a clearer error instead of reaching Unsloth's generic config=None error.

import gc
import os

# Safety re-assertion. This must normally be set before importing huggingface_hub,
# but keeping it here makes the intended state obvious in the notebook output.
os.environ['UNSLOTH_COMPILE_DISABLE'] = '1'
os.environ['UNSLOTH_COMPILE_IGNORE_ERRORS'] = '1'
os.environ['UNSLOTH_FULLGRAPH'] = '0'
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['TORCH_COMPILE_DISABLE'] = '1'
print('HF_HUB_ENABLE_HF_TRANSFER:', os.environ.get('HF_HUB_ENABLE_HF_TRANSFER'))
print('UNSLOTH_COMPILE_DISABLE:', os.environ.get('UNSLOTH_COMPILE_DISABLE'))
print('TORCHDYNAMO_DISABLE:', os.environ.get('TORCHDYNAMO_DISABLE'))
try:
    import torch._dynamo as _torch_dynamo
    _torch_dynamo.config.suppress_errors = True
    try:
        _torch_dynamo.disable()
    except Exception:
        pass
except Exception as exc:
    print('TorchDynamo disable check unavailable:', repr(exc))

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()


def verify_model_repo(repo_id, token=None):
    print('Checking model repo config:', repo_id)
    config_path = hf_hub_download(
        repo_id=repo_id,
        filename='config.json',
        token=token,
        force_download=False,
    )
    print('Config OK:', config_path)
    return config_path


# Prefer the Unsloth-specific quantized repo. Fall back to the other Unsloth 4-bit repo.
repo_candidates = []
for repo_id in [MEDGEMMA_MODEL_ID, FALLBACK_MEDGEMMA_MODEL_ID]:
    if repo_id and repo_id not in repo_candidates:
        repo_candidates.append(repo_id)

repo_errors = []
for repo_id in repo_candidates:
    try:
        verify_model_repo(repo_id, HF_TOKEN)
        MEDGEMMA_MODEL_ID = repo_id
        break
    except Exception as exc:
        repo_errors.append((repo_id, repr(exc)))
        print('Repo check failed:', repo_id)
        print(repr(exc))
else:
    raise RuntimeError(
        'Could not download config.json from any MedGemma repo. '
        'If you changed MEDGEMMA_MODEL_ID to the gated Google repo, confirm HF_TOKEN access. '
        f'Errors: {repo_errors}'
    )

print('Loading MedGemma with Unsloth from:', MEDGEMMA_MODEL_ID)

# Gemma 3 / MedGemma uses a SigLIP vision tower. In this Kaggle stack,
# Transformers may otherwise try torch flex_attention for the vision tower, but
# SiglipVisionModel rejects flex_attention. Use eager attention. Important:
# do NOT pass auto_config=... into Unsloth's FastModel/FastVisionModel loaders;
# Unsloth already passes auto_config internally, so passing it here causes:
#   TypeError: got multiple values for keyword argument 'auto_config'

def preview_eager_config(repo_id, token=None):
    cfg = AutoConfig.from_pretrained(
        repo_id,
        token=token,
        trust_remote_code=True,
    )
    cfg._attn_implementation = 'eager'
    if getattr(cfg, 'text_config', None) is not None:
        cfg.text_config._attn_implementation = 'eager'
    if getattr(cfg, 'vision_config', None) is not None:
        cfg.vision_config._attn_implementation = 'eager'
    print('Preview config attn:', getattr(cfg, '_attn_implementation', None))
    for attr in ('text_config', 'vision_config'):
        sub = getattr(cfg, attr, None)
        if sub is not None:
            print(f'Preview config.{attr} attn:', getattr(sub, '_attn_implementation', None))
    return cfg

_ = preview_eager_config(MEDGEMMA_MODEL_ID, HF_TOKEN)

COMMON_LOAD_KWARGS = dict(
    model_name=MEDGEMMA_MODEL_ID,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    load_in_4bit=True,
    token=HF_TOKEN,
    trust_remote_code=True,
    device_map='auto',
    use_gradient_checkpointing='unsloth',
    attn_implementation='eager',
    unsloth_force_compile=False,
)

UNSLOTH_API = FastVisionModel
UNSLOTH_LOADER_NAME = 'FastVisionModel'
try:
    model, processor = FastVisionModel.from_pretrained(**COMMON_LOAD_KWARGS)
except Exception as first_exc:
    print('FastVisionModel load failed. Retrying once with FastModel.')
    print('FastVisionModel error:', repr(first_exc))
    UNSLOTH_API = FastModel
    UNSLOTH_LOADER_NAME = 'FastModel'
    model, processor = FastModel.from_pretrained(**COMMON_LOAD_KWARGS)

print('Loaded with:', UNSLOTH_LOADER_NAME)
print('Processor/tokenizer type:', type(processor))

# Gemma 3 chat template for vision instruction tuning. If the repo already supplies a template, this is harmless.
try:
    processor = get_chat_template(processor, 'gemma-3')
except Exception as exc:
    print('get_chat_template failed; continuing with repo processor template:', repr(exc))

# Low-memory LoRA: attention adapters only, both vision + language.
# If this still OOMs, set finetune_vision_layers=False and rerun.
peft_kwargs = dict(
    finetune_vision_layers=FINETUNE_VISION_LAYERS,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias='none',
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

try:
    model = FastVisionModel.get_peft_model(model, **peft_kwargs)
except Exception as first_peft_exc:
    if hasattr(UNSLOTH_API, 'get_peft_model') and UNSLOTH_API is not FastVisionModel:
        print('FastVisionModel.get_peft_model failed. Retrying with loader API get_peft_model.')
        print('PEFT error:', repr(first_peft_exc))
        model = UNSLOTH_API.get_peft_model(model, **peft_kwargs)
    else:
        raise

def get_module_by_ending(model, ending):
    matches = [(name, module) for name, module in model.named_modules() if name.endswith(ending)]
    return matches[-1] if matches else (None, None)


def infer_vision_tower_dtype(model):
    """Return the dtype to feed pixel_values into SigLIP.

    Prefer a non-LayerNorm floating parameter because the vision tower can keep
    LayerNorm weights in float32 while the main SigLIP activations/weights are float16.
    """
    name, module = get_module_by_ending(model, 'vision_tower')
    if module is None:
        for n, m in model.named_modules():
            if 'vision_tower' in n.lower() or 'vision_model' in n.lower() or 'siglip' in n.lower():
                name, module = n, m
                break
    if module is None:
        print('Could not find vision tower; defaulting pixel dtype to float32.')
        return torch.float32

    # Prefer Conv/Linear/Embedding parameters over LayerNorm parameters.
    for sub_name, sub_module in module.named_modules():
        if isinstance(sub_module, (torch.nn.Conv2d, torch.nn.Linear, torch.nn.Embedding)):
            for p in sub_module.parameters(recurse=False):
                if torch.is_floating_point(p):
                    print('Vision tower dtype source:', f'{name}.{sub_name}', p.dtype)
                    return p.dtype
    for p in module.parameters():
        if torch.is_floating_point(p):
            print('Vision tower dtype source:', name, p.dtype)
            return p.dtype
    print('Vision tower has no floating parameters; defaulting pixel dtype to float32.')
    return torch.float32


def is_vision_module_name(name):
    lname = name.lower()
    return (
        'vision_tower' in lname
        or 'vision_model' in lname
        or 'siglip' in lname
        or '.vision.' in lname
    )


def disable_vision_gradient_checkpointing(model):
    """Disable SigLIP/Vision checkpointing even when the trainer uses GC for text.

    Unsloth's trainer wrapper calls model.for_training(...) at the start of train(),
    which can re-enable checkpointing. v26 also wraps model.for_training below so this
    function is run again after that call.
    """
    disabled = []
    for name, module in model.named_modules():
        if is_vision_module_name(name):
            if hasattr(module, 'gradient_checkpointing'):
                try:
                    module.gradient_checkpointing = False
                    disabled.append(name)
                except Exception:
                    pass
            if hasattr(module, 'gradient_checkpointing_disable'):
                try:
                    module.gradient_checkpointing_disable()
                    disabled.append(name + '.gradient_checkpointing_disable()')
                except Exception:
                    pass
    print(f'Vision gradient checkpointing disabled on {len(disabled)} modules/hooks.')
    if disabled:
        print('First disabled:', disabled[0])
        print('Last disabled:', disabled[-1])
    return disabled


def patch_siglip_layernorm_safe_fp32(model):
    """Patch SigLIP vision LayerNorms to compute in fp32 and return input dtype.

    Gemma 3 / MedGemma on T4 can keep some vision LayerNorm weights in float32 while
    the SigLIP activations are float16. Native torch.layer_norm requires matching
    dtypes, causing: RuntimeError: expected scalar type Half but found Float.

    This wrapper follows the usual mixed-precision LayerNorm pattern:
    compute LayerNorm in float32 for stability, then cast the output back to the
    activation dtype so downstream fp16 attention/MLP layers still receive fp16.
    """
    import types
    import torch.nn.functional as F

    patched = []
    already = []
    for name, module in model.named_modules():
        if is_vision_module_name(name) and isinstance(module, torch.nn.LayerNorm):
            if getattr(module, '_medgemma_safe_layernorm_patched', False):
                already.append(name)
                continue

            def _safe_forward(self, input):
                output_dtype = input.dtype
                weight = self.weight.float() if self.weight is not None else None
                bias = self.bias.float() if self.bias is not None else None
                output = F.layer_norm(
                    input.float(),
                    self.normalized_shape,
                    weight,
                    bias,
                    self.eps,
                )
                return output.to(dtype=output_dtype)

            safe_forward = types.MethodType(_safe_forward, module)
            # Accelerate device_map hooks replace module.forward with a wrapper that
            # calls module._old_forward. If that hook is present, patch _old_forward
            # so we keep Accelerate's device/offload behavior. Otherwise patch forward.
            if hasattr(module, '_old_forward'):
                module._old_forward = safe_forward
            else:
                module.forward = safe_forward
            module._medgemma_safe_layernorm_patched = True
            patched.append(name)

    print(f'SigLIP safe LayerNorm patch: {len(patched)} newly patched; {len(already)} already patched.')
    if patched:
        print('First safe LayerNorm:', patched[0])
        print('Last safe LayerNorm:', patched[-1])
    return patched + already


def force_full_vision_path_float32(model):
    """Emergency fallback: cast the full SigLIP vision path and projector to float32.

    This is heavier than the default pixel-dtype + safe-LayerNorm fix, but it avoids
    mixed Half/Float paths if a specific Kaggle/Unsloth stack keeps the vision
    activations in float32.
    """
    target_endings = ('vision_tower', 'multi_modal_projector')
    patched = []
    for ending in target_endings:
        name, module = get_module_by_ending(model, ending)
        if module is not None:
            module.to(torch.float32)
            patched.append(name)
    print(f'Full vision/projection float32 patch: {len(patched)} top-level modules cast.')
    for name in patched:
        print('  cast:', name)
    return patched


def apply_vision_runtime_fixes(model, where='manual'):
    print(f'Applying vision runtime fixes from: {where}')
    if PATCH_SIGLIP_LAYERNORM_SAFE_FP32:
        patch_siglip_layernorm_safe_fp32(model)
    if DISABLE_VISION_GRADIENT_CHECKPOINTING:
        disable_vision_gradient_checkpointing(model)
    if FORCE_FULL_VISION_FLOAT32:
        force_full_vision_path_float32(model)
    return model


def wrap_model_for_training_to_reapply_vision_fixes(model):
    """UnslothSFTTrainer calls model.for_training(...) inside train().

    That mode switch can undo vision checkpointing changes. Wrap it once so the
    SigLIP fixes are applied immediately after every for_training call.
    """
    import types

    if not hasattr(model, 'for_training'):
        print('model.for_training not found; no wrapping needed.')
        return False
    if getattr(model, '_medgemma_for_training_wrapped', False):
        print('model.for_training already wrapped.')
        return True

    original_for_training = model.for_training

    def _wrapped_for_training(self, *args, **kwargs):
        result = original_for_training(*args, **kwargs)
        apply_vision_runtime_fixes(self, where='model.for_training wrapper')
        return result

    model.for_training = types.MethodType(_wrapped_for_training, model)
    model._medgemma_for_training_wrapped = True
    print('Wrapped model.for_training to re-apply SigLIP vision fixes during trainer.train().')
    return True


# Apply now, and wrap model.for_training so Trainer cannot undo the vision fixes.
apply_vision_runtime_fixes(model, where='after PEFT load')
if PATCH_FOR_TRAINING_REAPPLY_VISION_FIXES:
    wrap_model_for_training_to_reapply_vision_fixes(model)

VISION_INPUT_DTYPE = infer_vision_tower_dtype(model)
print('Pixel values will be cast to:', VISION_INPUT_DTYPE if CAST_PIXEL_VALUES_TO_VISION_DTYPE else 'collator default')

print('Trainable parameters:')
model.print_trainable_parameters()
if torch.cuda.is_available():
    print('GPU allocated GB after model load:', torch.cuda.memory_allocated() / 1024**3)
    print('GPU reserved GB after model load:', torch.cuda.memory_reserved() / 1024**3)


## 9. Train MedGemma QLoRA with Unsloth


In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint

# TorchDynamo pre-train guard: Unsloth's Gemma patches may otherwise try to compile q_norm/prepare.
try:
    import torch._dynamo as _torch_dynamo
    _torch_dynamo.config.suppress_errors = True
    try:
        _torch_dynamo.disable()
    except Exception:
        pass
    print('TorchDynamo pre-train guard active.')
except Exception as exc:
    print('TorchDynamo pre-train guard unavailable:', repr(exc))

(UNSLOTH_API if 'UNSLOTH_API' in globals() else FastVisionModel).for_training(model)

# Re-apply after for_training, because mode switches can re-enable checkpointing
# and can recreate mixed precision assumptions in the vision path.
apply_vision_runtime_fixes(model, where='before trainer setup')
if PATCH_FOR_TRAINING_REAPPLY_VISION_FIXES:
    wrap_model_for_training_to_reapply_vision_fixes(model)

VISION_INPUT_DTYPE = infer_vision_tower_dtype(model)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()


class SafeGemma3VisionDataCollator:
    """Thin wrapper around UnslothVisionDataCollator for Gemma3/SigLIP dtype stability.

    The previous crash happened in SigLIP LayerNorm: activations and weights were
    mixed Half/Float. v26 also patches SigLIP LayerNorm forward passes to
    compute in float32 and return the input activation dtype.
    """

    def __init__(self, base_collator, target_pixel_dtype=None):
        self.base_collator = base_collator
        self.target_pixel_dtype = target_pixel_dtype

    def __call__(self, examples):
        batch = self.base_collator(examples)
        if self.target_pixel_dtype is not None and 'pixel_values' in batch:
            pixel_values = batch['pixel_values']
            if torch.is_tensor(pixel_values) and torch.is_floating_point(pixel_values):
                batch['pixel_values'] = pixel_values.to(dtype=self.target_pixel_dtype)
        return batch


base_collator = UnslothVisionDataCollator(model, processor)
target_pixel_dtype = VISION_INPUT_DTYPE if CAST_PIXEL_VALUES_TO_VISION_DTYPE else None
print('Data collator target pixel dtype:', target_pixel_dtype)
data_collator = SafeGemma3VisionDataCollator(
    base_collator,
    target_pixel_dtype=target_pixel_dtype,
)

# Small preflight so dtype problems appear before the full trainer loop.
try:
    preview_batch = data_collator([train_conversations[0]])
    print('Preview batch tensor dtypes:')
    for key, value in preview_batch.items():
        if torch.is_tensor(value):
            print(f'  {key}: shape={tuple(value.shape)}, dtype={value.dtype}, device={value.device}')
    del preview_batch
except Exception as exc:
    print('Preflight collator check failed:')
    print(repr(exc))
    raise

max_steps_arg = MAX_STEPS if MAX_STEPS is not None else -1
num_train_epochs_arg = NUM_TRAIN_EPOCHS if MAX_STEPS is None else 1
TRAINER_OUTPUT_DIR = PROJECT_DIR / 'trainer_outputs'
TRAINER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_conversations,
    processing_class=processor.tokenizer,
    data_collator=data_collator,
    args=SFTConfig(
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
        gradient_checkpointing_kwargs={'use_reentrant': False} if USE_GRADIENT_CHECKPOINTING else None,
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        max_steps=max_steps_arg,
        num_train_epochs=num_train_epochs_arg,
        learning_rate=LEARNING_RATE,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_strategy='steps',
        save_total_limit=2,
        optim='adamw_torch_fused',
        weight_decay=0.001,
        lr_scheduler_type='cosine',
        seed=SEED,
        output_dir=str(TRAINER_OUTPUT_DIR),
        report_to='none',
        # Do not set fp16=True/bf16=False here. Unsloth handles Gemma 3 dtype internally;
        # forcing Trainer fp16 can produce SigLIP LayerNorm Half/Float mismatches on T4.
        remove_unused_columns=False,
        dataset_text_field='',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_length=MAX_SEQUENCE_LENGTH,
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
    ),
)

last_checkpoint = get_last_checkpoint(str(TRAINER_OUTPUT_DIR))
if last_checkpoint:
    print('Resuming trainer from checkpoint:', last_checkpoint)
else:
    print('No trainer checkpoint found; starting from step 0.')

start = time.time()
train_result = trainer.train(resume_from_checkpoint=last_checkpoint if last_checkpoint else None)
train_minutes = (time.time() - start) / 60
print(f'Training finished in {train_minutes:.2f} minutes')
print(train_result)

with open(RESULTS_DIR / 'train_result.json', 'w') as f:
    json.dump({
        'train_minutes': train_minutes,
        'train_result': str(train_result),
        'model_id': MEDGEMMA_MODEL_ID,
        'unsloth_loader': globals().get('UNSLOTH_LOADER_NAME', 'FastVisionModel'),
        'train_per_class': TRAIN_PER_CLASS,
        'max_steps': MAX_STEPS,
        'learning_rate': LEARNING_RATE,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'max_sequence_length': MAX_SEQUENCE_LENGTH,
        'finetune_vision_layers': FINETUNE_VISION_LAYERS,
        'cast_pixel_values_to_vision_dtype': CAST_PIXEL_VALUES_TO_VISION_DTYPE,
        'disable_vision_gradient_checkpointing': DISABLE_VISION_GRADIENT_CHECKPOINTING,
        'force_full_vision_float32': FORCE_FULL_VISION_FLOAT32,
        'patch_siglip_layernorm_safe_fp32': PATCH_SIGLIP_LAYERNORM_SAFE_FP32,
        'patch_for_training_reapply_vision_fixes': PATCH_FOR_TRAINING_REAPPLY_VISION_FIXES,
        'disable_unsloth_compile': DISABLE_UNSLOTH_COMPILE,
        'use_gradient_checkpointing': USE_GRADIENT_CHECKPOINTING,
        'vision_input_dtype': str(VISION_INPUT_DTYPE),
    }, f, indent=2)


## 10. Save adapters


In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))
print('Saved LoRA adapter and processor to:', ADAPTER_DIR)

adapter_zip = shutil.make_archive(str(PROJECT_DIR / 'medgemma_crc_unsloth_lora_adapter'), 'zip', ADAPTER_DIR)
print('Adapter zip:', adapter_zip)


## 11. Forced-choice digit scoring helpers


In [ ]:
(UNSLOTH_API if 'UNSLOTH_API' in globals() else FastVisionModel).for_inference(model)
model.eval()


def move_inputs_to_device(inputs, device=DEVICE):
    moved = {}
    for k, v in inputs.items():
        if torch.is_tensor(v):
            v = v.to(device)
            # P100 + Gemma 3 vision is more stable with float32 image tensors.
            if k in ('pixel_values', 'images') and torch.is_floating_point(v):
                v = v.to(torch.float32)
        moved[k] = v
    return moved


def build_prompt_inputs(image):
    if hasattr(image, 'convert'):
        image = image.convert('RGB')

    # Preferred Transformers multimodal path: put the PIL image directly in the chat message.
    messages_with_image = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': CLASS_INSTRUCTION},
            ],
        }
    ]
    try:
        inputs = processor.apply_chat_template(
            messages_with_image,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors='pt',
        )
        return move_inputs_to_device(inputs)
    except Exception as exc:
        print('apply_chat_template tokenized path failed once; falling back to processor(image, text). Error:', repr(exc))

    # Fallback path used by older Unsloth vision examples.
    messages_text_only = [
        {
            'role': 'user',
            'content': [
                {'type': 'text', 'text': CLASS_INSTRUCTION},
                {'type': 'image'},
            ],
        }
    ]
    input_text = processor.apply_chat_template(messages_text_only, add_generation_prompt=True)
    inputs = processor(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors='pt',
    )
    return move_inputs_to_device(inputs)


def get_digit_token_ids(tokenizer):
    mapping = {}
    for digit in [str(i) for i in range(9)]:
        variants = [digit, ' ' + digit, '\n' + digit]
        ids = []
        for v in variants:
            toks = tokenizer(v, add_special_tokens=False).input_ids
            if len(toks) == 1:
                ids.append((v, toks[0]))
        if not ids:
            raise RuntimeError(f'Could not find a single-token encoding for digit {digit}.')
        mapping[digit] = ids
    return mapping

DIGIT_TOKEN_IDS = get_digit_token_ids(processor.tokenizer)
print('Digit token IDs:')
for digit, ids in DIGIT_TOKEN_IDS.items():
    print(digit, ids)


def predict_digit_for_image(image):
    inputs = build_prompt_inputs(image)
    with torch.inference_mode():
        outputs = model(**inputs, use_cache=False, return_dict=True)
        logits = outputs.logits[:, -1, :].float()
        if not torch.isfinite(logits).all():
            raise RuntimeError('Non-finite logits detected during prediction. Stop and lower LR or verify dtype settings.')
        log_probs = torch.log_softmax(logits, dim=-1)[0]

    digit_scores = {}
    for digit, variants in DIGIT_TOKEN_IDS.items():
        # Use the best score among no-space, leading-space, and newline variants.
        scores = [float(log_probs[token_id].item()) for _, token_id in variants]
        digit_scores[digit] = max(scores)

    best_digit = max(digit_scores, key=digit_scores.get)
    pred_label = DIGIT_TO_LABEL[best_digit]
    top = sorted(digit_scores.items(), key=lambda kv: kv[1], reverse=True)
    raw = 'forced_digit_top9=' + ', '.join(f'{DIGIT_TO_LABEL[d]}:{s:.3f}' for d, s in top)
    return pred_label, raw, digit_scores


## 12. Smoke test before full evaluation


In [ ]:
smoke_n = min(24, len(test_dataset))
smoke_rows = []
print(f'Running smoke test on {smoke_n} samples...')
for i in range(smoke_n):
    sample = test_dataset[i]
    true_label = label_name_for_sample(sample, test_label_names)
    pred_label, raw, scores = predict_digit_for_image(sample['image'])
    smoke_rows.append({'sample_index': i, 'true_label': true_label, 'predicted_label': pred_label, 'raw_scores': raw})
    print(f'{i:02d} true={true_label:>5s} pred={pred_label:>5s} {raw[:100]}')

smoke_df = pd.DataFrame(smoke_rows)
smoke_df.to_csv(RESULTS_DIR / 'smoke_test_predictions.csv', index=False)
print('Smoke accuracy:', simple_accuracy(smoke_df.true_label, smoke_df.predicted_label))
print('Predicted classes:', sorted(smoke_df.predicted_label.unique().tolist()))

if smoke_df.predicted_label.nunique() <= 1:
    raise RuntimeError('Smoke test collapsed to one class. Do not run full evaluation yet. Try increasing MAX_STEPS or check training loss.')


## 13. Evaluate on CRC-VAL-HE-7K with checkpoints


In [ ]:
eval_dataset = test_dataset
if EVAL_MAX_SAMPLES is not None:
    eval_dataset = eval_dataset.select(range(min(EVAL_MAX_SAMPLES, len(eval_dataset))))
TOTAL_EVAL = len(eval_dataset)
print('Evaluation samples:', TOTAL_EVAL)

PREDICTIONS_PATH = RESULTS_DIR / 'medgemma_unsloth_qlora_predictions.csv'
CHECKPOINT_PATH = RESULTS_DIR / 'medgemma_unsloth_qlora_predictions_checkpoint.csv'

prediction_rows = []
if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)
    prediction_rows = checkpoint_df.to_dict('records')
    processed_indices = set(checkpoint_df['sample_index'].astype(int).tolist()) if len(checkpoint_df) else set()
    print(f'Loaded checkpoint with {len(prediction_rows)} rows. Resuming from {max(processed_indices) + 1 if processed_indices else 0}/{TOTAL_EVAL}.')
else:
    processed_indices = set()
    print('No checkpoint found. Starting evaluation from 0.')

start_time = time.time()
last_checkpoint_time = time.time()

for i in range(TOTAL_EVAL):
    if i in processed_indices:
        continue
    if i % 25 == 0:
        print(f'Processing sample {i}/{TOTAL_EVAL}')

    sample = eval_dataset[i]
    true_label = label_name_for_sample(sample, test_label_names)
    pred_label, raw, digit_scores = predict_digit_for_image(sample['image'])

    row = {
        'sample_index': i,
        'true_label': true_label,
        'predicted_label': pred_label,
        'raw_scores': raw,
    }
    for digit, score in digit_scores.items():
        row[f'score_{DIGIT_TO_LABEL[digit]}'] = score
    prediction_rows.append(row)

    if (i + 1) % EVAL_CHECKPOINT_EVERY == 0:
        pd.DataFrame(prediction_rows).to_csv(CHECKPOINT_PATH, index=False)
        now = time.time()
        checkpoint_minutes = (now - last_checkpoint_time) / 60
        last_checkpoint_time = now
        samples_done = i + 1
        samples_left = TOTAL_EVAL - samples_done
        samples_per_minute = EVAL_CHECKPOINT_EVERY / checkpoint_minutes if checkpoint_minutes > 0 else float('inf')
        eta_hours = samples_left / samples_per_minute / 60 if samples_per_minute > 0 else float('inf')
        print(f'Saved checkpoint: {samples_done}/{TOTAL_EVAL} -> {CHECKPOINT_PATH}')
        print(f'Speed: {samples_per_minute:.2f} samples/min | ETA: {eta_hours:.2f} hours')

    if torch.cuda.is_available() and (i + 1) % 25 == 0:
        torch.cuda.empty_cache()

results_df = pd.DataFrame(prediction_rows).sort_values('sample_index')
results_df.to_csv(PREDICTIONS_PATH, index=False)
results_df.to_csv(CHECKPOINT_PATH, index=False)
print('Saved predictions:', PREDICTIONS_PATH)
print('Evaluation minutes:', (time.time() - start_time) / 60)
results_df.head()


## 14. Metrics, reports, confusion matrix, and zip


In [ ]:
results_df = pd.read_csv(RESULTS_DIR / 'medgemma_unsloth_qlora_predictions.csv')
y_true = results_df['true_label'].tolist()
y_pred = results_df['predicted_label'].tolist()

accuracy = simple_accuracy(y_true, y_pred)
report = classification_report_df(y_true, y_pred, LABEL_NAMES)
cm = simple_confusion_matrix(y_true, y_pred, LABEL_NAMES)

metrics = {
    'model': 'medgemma_unsloth_qlora_digit_target_lowmem_v26',
    'base_model': MEDGEMMA_MODEL_ID,
    'kaggle_gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'num_eval_samples': int(len(results_df)),
    'accuracy': accuracy,
    'macro_f1': float(report.loc[report['label'] == 'macro avg', 'f1-score'].iloc[0]),
    'weighted_f1': float(report.loc[report['label'] == 'weighted avg', 'f1-score'].iloc[0]),
    'train_per_class': TRAIN_PER_CLASS,
    'max_steps': MAX_STEPS,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'learning_rate': LEARNING_RATE,
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'eval_max_samples': EVAL_MAX_SAMPLES,
}

with open(RESULTS_DIR / 'medgemma_unsloth_qlora_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
report.to_csv(RESULTS_DIR / 'medgemma_unsloth_qlora_classification_report.csv', index=False)
pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES).to_csv(RESULTS_DIR / 'medgemma_unsloth_qlora_confusion_matrix.csv')

plot_confusion_matrix(cm, LABEL_NAMES, 'MedGemma Unsloth QLoRA Confusion Matrix', RESULTS_DIR / 'medgemma_unsloth_qlora_confusion_matrix.png')

print(json.dumps(metrics, indent=2))
print(report)

# Save a comparison template. Add your ViT numbers manually if needed.
comparison = pd.DataFrame([
    {'Model': 'Previous MedGemma QLoRA', 'Accuracy': 0.26016713091922006, 'Notes': 'Uploaded previous run'},
    {'Model': 'MedGemma Unsloth QLoRA digit target lowmem v22', 'Accuracy': accuracy, 'Notes': f'NCT balanced train_per_class={TRAIN_PER_CLASS}, max_steps={MAX_STEPS}, eval_max={EVAL_MAX_SAMPLES}'},
])
comparison.to_csv(RESULTS_DIR / 'comparison_summary.csv', index=False)

zip_path = shutil.make_archive(str(PROJECT_DIR / 'medgemma_unsloth_qlora_results'), 'zip', RESULTS_DIR)
print('Created results zip:', zip_path)

# Kaggle FileLink sometimes 404s for absolute paths. This gives a relative link from /kaggle/working.
os.chdir('/kaggle/working')
try:
    from IPython.display import FileLink, display
    display(FileLink('MedGemma_Unsloth_QLoRA_CRC_lowmem_v26/medgemma_unsloth_qlora_results.zip'))
except Exception as exc:
    print('FileLink unavailable:', exc)
    print('Download manually from:', zip_path)


## 15. Optional: save a committed Kaggle output


In [ ]:
print('Important: before ending the session, download this zip or click Save Version / Commit in Kaggle:')
print('/kaggle/working/MedGemma_Unsloth_QLoRA_CRC_lowmem_v26/medgemma_unsloth_qlora_results.zip')
print('Adapter zip:')
print('/kaggle/working/MedGemma_Unsloth_QLoRA_CRC_lowmem_v26/medgemma_crc_unsloth_lora_adapter.zip')
